In [0]:
import logging
from pyspark.sql import DataFrame

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
configs = parse_config_from_json("/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/configs.json")

In [0]:
logger = get_logger()

In [0]:
def json_reader(file_path: str, multi_line: bool = True) -> DataFrame:
    """ Read a JSON file into a DataFrame """
    try:
        logger.info("Reading JSON: %s (multi_line=%s)", file_path, multi_line)
        if multi_line:
            df = (
                spark.read.format("json")
                .option("multiLine", multi_line)
                .load(file_path)
            )
        else:
            df = spark.read.json(file_path)
        logger.info("JSON read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"JSON file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read JSON '{file_path}': {e}") from e

In [0]:
def csv_reader(file_path: str, delimiter: str = ",", header: bool = True) -> DataFrame:
    """ Read a CSV file into a DataFrame """
    try:
        logger.info("Reading CSV: %s (delimiter='%s', header=%s)", file_path, delimiter, header)
        df = (
            spark.read.format("csv")
            .options(header=header, delimiter=delimiter, escape='"')
            .load(file_path)
        )
        logger.info("CSV read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"CSV file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read CSV '{file_path}': {e}")

In [0]:
def excel_reader(file_path: str, sheet_name: str) -> DataFrame:
    """Read an Excel sheet into a DataFrame."""
    try:
        logger.info("Reading Excel: %s (sheet='%s')", file_path, sheet_name)
        df = (
            spark.read.format("excel")
            .option("headerRows", 1)
            .option("inferSchema", "true")
            .load(file_path, sheetName=sheet_name)
        )
        logger.info("Excel read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"Excel file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read Excel '{file_path}': {e}") from e

In [0]:
def apply_column_alias(df: DataFrame, schema_dict: dict) -> DataFrame:
    """Rename columns according to a mapping dictionary."""
    try:
        aliases = schema_dict
        logger.info("Applying %d column aliases", len(aliases))
        return df.select(
            [df[col].alias(aliases[col]) if col in aliases else df[col] for col in df.columns]
        )
    except Exception as e:
        logger.error("Failed to apply column alias: %s", e)

In [0]:
def _ingestion_util(module: str):
    try:
        try:
            conf = configs.get(module)
        except KeyError:
            raise Exception(f"Module {module} not found in config file")

        schemas = conf["schema_alias"]
        src_path = conf["file_path"]
        target_table = conf["target_table"]
        src_format = conf["file_format"]

        if src_format == "json":
            df = json_reader(src_path)
        elif src_format == "csv":
            df = csv_reader(src_path)
        elif src_format == "excel":
            sheet_name = conf["params"]["sheet_name"]
            df = excel_reader(src_path, sheet_name)
        else:
            raise Exception("Invalid file format")
        
        # Apply schema alias
        df = apply_column_alias(df, schemas)

        # write bronze tables 
        delta_writer(df, target_table)

    except Exception as e:
        logger.error(f"Error ingesting {module}: {e}")

In [0]:
def main_ingestion():
    ingestion_candidates = list(configs.keys())
    logger.info("Ingesting modules - %s", str(ingestion_candidates))
    for module in ingestion_candidates:
        try:
            _ingestion_util(module)
        except Exception as e:
            logger.error(f"Error in main while ingesting {module}: {e}")  